In [ ]:
# Cell 1 — imports e carregamento
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

df = pd.read_csv('../data/processed/crime_2020_2025_clean.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])
df['hour'] = df['timestamp'].dt.hour
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

In [ ]:
# Cell 2 — crimes por área (distrito LAPD)
by_area = df.groupby('area_name').size().reset_index(name='total_crimes')
by_area = by_area.sort_values('total_crimes', ascending=False)
print(by_area.to_string())

plt.figure(figsize=(12, 5))
plt.bar(by_area['area_name'], by_area['total_crimes'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Total de Crimes por Distrito LAPD (2020-2025)')
plt.ylabel('Ocorrências')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3 — distribuição por hora do dia
by_hour = df.groupby('hour').size().reset_index(name='count')

plt.figure(figsize=(12, 4))
plt.plot(by_hour['hour'], by_hour['count'], marker='o', color='tomato')
plt.title('Crimes por Hora do Dia')
plt.xlabel('Hora')
plt.ylabel('Ocorrências')
plt.xticks(range(0, 24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — crimes por área E hora (tabela de calor)
pivot = df.groupby(['area_name', 'hour']).size().unstack(fill_value=0)

plt.figure(figsize=(16, 8))
plt.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
plt.colorbar(label='Ocorrências')
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(24))
plt.xlabel('Hora')
plt.title('Heatmap: Crimes por Área e Hora')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — percentis da distribuição de peso
# Simula o que o backend calcula para o heatmap
counts = by_area['total_crimes'].values
min_c, max_c = counts.min(), counts.max()
weights = (counts - min_c) / (max_c - min_c)

print('Distribuição dos pesos normalizados (0-1):')
for p in [10, 25, 50, 75, 90, 95, 99, 100]:
    print(f'  p{p:3d}: {np.percentile(weights, p):.3f}')

print(f'\nÁreas com peso > 0.85 (vermelho no heatmap):')
for area, w in zip(by_area['area_name'], weights):
    if w > 0.85:
        print(f'  {area}: {w:.3f}')

print(f'\nÁreas com peso 0.60-0.85 (laranja):')
for area, w in zip(by_area['area_name'], weights):
    if 0.60 <= w <= 0.85:
        print(f'  {area}: {w:.3f}')

In [ ]:
# Cell 6 — recomendação de thresholds
print('=== RECOMENDAÇÃO DE THRESHOLDS PARA O HEATMAP ===\n')
print('Com base na distribuição real dos dados:\n')

thresholds = {
    'vermelho (Alto)': np.percentile(weights, 85),
    'laranja (Médio-Alto)': np.percentile(weights, 60),
    'amarelo (Médio)': np.percentile(weights, 35),
    'verde (Baixo)': 0.0
}

for label, val in thresholds.items():
    print(f'  {label}: weight > {val:.3f}')

print('\nCola esses valores no Flutter em _heatmapColor()')